In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\sanjay\OneDrive\Desktop\CS50\005Customer Personality Analysis\marketing_campaign.csv", sep="\t")

In [2]:
print(df.shape)
print(df.columns)
df.head()

(2240, 29)
Index(['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome',
       'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits',
       'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts',
       'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases',
       'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth',
       'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1',
       'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response'],
      dtype='str')


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,04-09-2012,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,08-03-2014,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,10-02-2014,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


In [3]:
print(df.isnull().sum())

ID                      0
Year_Birth              0
Education               0
Marital_Status          0
Income                 24
Kidhome                 0
Teenhome                0
Dt_Customer             0
Recency                 0
MntWines                0
MntFruits               0
MntMeatProducts         0
MntFishProducts         0
MntSweetProducts        0
MntGoldProds            0
NumDealsPurchases       0
NumWebPurchases         0
NumCatalogPurchases     0
NumStorePurchases       0
NumWebVisitsMonth       0
AcceptedCmp3            0
AcceptedCmp4            0
AcceptedCmp5            0
AcceptedCmp1            0
AcceptedCmp2            0
Complain                0
Z_CostContact           0
Z_Revenue               0
Response                0
dtype: int64


In [4]:
missing_counts = df.isnull().sum()
print(missing_counts[missing_counts > 0])

Income    24
dtype: int64


In [5]:
# Initialize an empty summary dataframe with column names as index
summary_df = pd.DataFrame(index=df.columns)

# Calculate missing values per column
summary_df["Missing Values"] = df.isnull().sum()

# Calculate duplicate values per column using a list comprehension
summary_df["Duplicate Values"] = [df[col].duplicated().sum() for col in df.columns]

# Isolate and display columns that have any missing or duplicate values
filtered_summary = summary_df[(summary_df["Missing Values"] > 0) | (summary_df["Duplicate Values"] > 0)]
print(filtered_summary)

                     Missing Values  Duplicate Values
Year_Birth                        0              2181
Education                         0              2235
Marital_Status                    0              2232
Income                           24               265
Kidhome                           0              2237
Teenhome                          0              2237
Dt_Customer                       0              1577
Recency                           0              2140
MntWines                          0              1464
MntFruits                         0              2082
MntMeatProducts                   0              1682
MntFishProducts                   0              2058
MntSweetProducts                  0              2063
MntGoldProds                      0              2027
NumDealsPurchases                 0              2225
NumWebPurchases                   0              2225
NumCatalogPurchases               0              2226
NumStorePurchases           

In [6]:
# Check for duplicate entries based strictly on the unique ID column
print("Duplicate rows based on ID:", df.duplicated(subset=["ID"]).sum())

# Check for identical duplicate rows across all columns combined
print("Completely identical rows:", df.duplicated().sum())

print("Duplicate records with both name of row and values :", df[df.duplicated()])

Duplicate rows based on ID: 0
Completely identical rows: 0
Duplicate records with both name of row and values : Empty DataFrame
Columns: [ID, Year_Birth, Education, Marital_Status, Income, Kidhome, Teenhome, Dt_Customer, Recency, MntWines, MntFruits, MntMeatProducts, MntFishProducts, MntSweetProducts, MntGoldProds, NumDealsPurchases, NumWebPurchases, NumCatalogPurchases, NumStorePurchases, NumWebVisitsMonth, AcceptedCmp3, AcceptedCmp4, AcceptedCmp5, AcceptedCmp1, AcceptedCmp2, Complain, Z_CostContact, Z_Revenue, Response]
Index: []

[0 rows x 29 columns]


In [7]:
print(df["Income"].values)

[58138. 46344. 71613. ... 56981. 69245. 52869.]


In [8]:
# Calculate and apply the median income based on matching Education and Marital_Status
df["Income"] = df.groupby(["Education", "Marital_Status"])["Income"].transform(
    lambda x: x.fillna(x.median())
)

# Check if any missing values remain
print("Remaining missing values:", df["Income"].isnull().sum())

Remaining missing values: 0


In [9]:
for col in ["Education", "Marital_Status"]:
    df[col] = df[col].astype(str).str.strip().str.title()

# Verify unique string values now align uniformly
print(df["Education"].unique())
print(df["Marital_Status"].unique())

<ArrowStringArray>
['Graduation', 'Phd', 'Master', 'Basic', '2N Cycle']
Length: 5, dtype: str
<ArrowStringArray>
['Single', 'Together', 'Married', 'Divorced', 'Widow', 'Alone', 'Absurd',
 'Yolo']
Length: 8, dtype: str


In [10]:
# Standardize marital statuses into uniform corporate designations
marital_map = {
    "Single": "Single",
    "Together": "Partnered",
    "Married": "Partnered",
    "Divorced": "Divorced",
    "Widow": "Widowed",
    "Alone": "Single",
    "Absurd": "Single",
    "Yolo": "Single"
}

df["Marital_Status"] = df["Marital_Status"].map(marital_map)
print(df["Marital_Status"].value_counts())

Marital_Status
Partnered    1444
Single        487
Divorced      232
Widowed        77
Name: count, dtype: int64


In [11]:
# Map non-standard education entries to universal terms
education_map = {
    "Graduation": "Graduate",
    "Phd": "PhD",
    "Master": "Postgraduate",
    "Basic": "Undergraduate",
    "2N Cycle": "Postgraduate"
}

df["Education"] = df["Education"].map(education_map)
print(df["Education"].value_counts())

Education
Graduate         1127
Postgraduate      573
PhD               486
Undergraduate      54
Name: count, dtype: int64


In [12]:
# Convert Dt_Customer column to standard datetime format
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")

# Verify the data type conversion
print(df["Dt_Customer"].dtype)

datetime64[us]


In [13]:
# Drop non-informative constant columns out of the dataset permanently
df.drop(columns=["Z_CostContact", "Z_Revenue"], inplace=True, errors="ignore")

# Convert Dt_Customer column to a clear, clean datetime type
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")

# Verify updates directly
print("New total columns remaining:", len(df.columns))
print("Dt_Customer updated data type:", df["Dt_Customer"].dtype)

New total columns remaining: 27
Dt_Customer updated data type: datetime64[us]


In [14]:
# Check for impossible baseline metrics in birth years
print("Oldest customers recorded:")
print(df["Year_Birth"].sort_values().head(5))

# Filter out extreme birth year errors (keeping realistic ages under 100)
df = df[df["Year_Birth"] > 1926]

# Verify the updated demographic baseline range
print("\nCleaned minimum birth year:", df["Year_Birth"].min())

Oldest customers recorded:
239     1893
339     1899
192     1900
1950    1940
424     1941
Name: Year_Birth, dtype: int64

Cleaned minimum birth year: 1940


In [15]:
# Calculate customer age at the point of enrollment
df["Age"] = df["Dt_Customer"].dt.year - df["Year_Birth"]

# Identify and filter extreme age outliers (e.g., ages > 100)
print("Ages before cleaning:", df["Age"].sort_values(ascending=False).head(5))
df = df[df["Age"] < 100]

Ages before cleaning: 1950    73
424     72
415     71
1923    70
2084    70
Name: Age, dtype: int64


In [16]:
# Combine individual kid and teen columns into a uniform Dependant Count field
df["Dependents"] = df["Kidhome"] + df["Teenhome"]

# Drop original individual metrics to keep the final export clean
df.drop(columns=["Kidhome", "Teenhome"], inplace=True)

In [17]:
# Display summary of final data types and non-null counts
print(df.info())
print("\nFinal shape:", df.shape)

<class 'pandas.DataFrame'>
Index: 2237 entries, 0 to 2239
Data columns (total 27 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   ID                   2237 non-null   int64         
 1   Year_Birth           2237 non-null   int64         
 2   Education            2237 non-null   str           
 3   Marital_Status       2237 non-null   str           
 4   Income               2237 non-null   float64       
 5   Dt_Customer          2237 non-null   datetime64[us]
 6   Recency              2237 non-null   int64         
 7   MntWines             2237 non-null   int64         
 8   MntFruits            2237 non-null   int64         
 9   MntMeatProducts      2237 non-null   int64         
 10  MntFishProducts      2237 non-null   int64         
 11  MntSweetProducts     2237 non-null   int64         
 12  MntGoldProds         2237 non-null   int64         
 13  NumDealsPurchases    2237 non-null   int64       

In [18]:
# 1. Clean extreme age outliers safely using a logical mask
df = df[df["Year_Birth"] > 1920]

# 2. Recalculate Age to ensure absolute accuracy against enrollment year
df["Age"] = df["Dt_Customer"].dt.year - df["Year_Birth"]

# 3. Create Dependents and drop source columns safely using errors='ignore'
if "Kidhome" in df.columns and "Teenhome" in df.columns:
    df["Dependents"] = df["Kidhome"] + df["Teenhome"]
df.drop(columns=["Kidhome", "Teenhome"], inplace=True, errors="ignore")

# 4. Final Data Integrity Inspection
print("Updated Shape:", df.shape)
print("\nOldest check (Min Birth Year):", df["Year_Birth"].min())
print("Maximum calculated Age:", df["Age"].max())

Updated Shape: (2237, 27)

Oldest check (Min Birth Year): 1940
Maximum calculated Age: 73


In [19]:
df.head(29)

,ID,Year_Birth,Education,Marital_Status,Income,Dt_Customer,Recency,MntWines,MntFruits,MntMeatProducts,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Response,Age,Dependents
0,5524,1957,Graduate,Single,58138.0,2012-09-04,58,635,88,546,...,7,0,0,0,0,0,0,1,55,0
1,2174,1954,Graduate,Single,46344.0,2014-03-08,38,11,1,6,...,5,0,0,0,0,0,0,0,60,2
2,4141,1965,Graduate,Partnered,71613.0,2013-08-21,26,426,49,127,...,4,0,0,0,0,0,0,0,48,0
3,6182,1984,Graduate,Partnered,26646.0,2014-02-10,26,11,4,20,...,6,0,0,0,0,0,0,0,30,1
4,5324,1981,PhD,Partnered,58293.0,2014-01-19,94,173,43,118,...,5,0,0,0,0,0,0,0,33,1
5,7446,1967,Postgraduate,Partnered,62513.0,2013-09-09,16,520,42,98,...,6,0,0,0,0,0,0,0,46,1
6,965,1971,Graduate,Divorced,55635.0,2012-11-13,34,235,65,164,...,6,0,0,0,0,0,0,0,41,1
7,6177,1985,PhD,Partnered,33454.0,2013-05-08,32,76,10,56,...,8,0,0,0,0,0,0,0,28,1
8,4855,1974,PhD,Partnered,30351.0,2013-06-06,19,14,0,24,...,9,0,0,0,0,0,0,1,39,1
9,5899,1950,PhD,Partnered,5648.0,2014-03-13,68,28,0,6,...,20,1,0,0,0,0,0,0,64,2


In [21]:
print(df.isnull().sum())

ID                     0
Year_Birth             0
Education              0
Marital_Status         0
Income                 0
Dt_Customer            0
Recency                0
MntWines               0
MntFruits              0
MntMeatProducts        0
MntFishProducts        0
MntSweetProducts       0
MntGoldProds           0
NumDealsPurchases      0
NumWebPurchases        0
NumCatalogPurchases    0
NumStorePurchases      0
NumWebVisitsMonth      0
AcceptedCmp3           0
AcceptedCmp4           0
AcceptedCmp5           0
AcceptedCmp1           0
AcceptedCmp2           0
Complain               0
Response               0
Age                    0
Dependents             0
dtype: int64


In [22]:
# Save the clean production data to a new CSV file without the internal index
df.to_csv("clean_customer_campaign_data.csv", index=False)
print("Data export successful. Ready for import.")

Data export successful. Ready for import.


Performed missing value imputation, demographic validation, category standardization, text normalization, duplicate auditing, date verification, datatype correction, and customer profile enrichment. Final dataset contains 2,237 records with zero missing values, standardized categorical fields, validated customer ages, cleaned text attributes, and CRM/Power BI ready formatting.